# Turkey City District Code Table (Central Primary Key)

Builds the single reference table `city_code | city_name | district_code | district_name` from the two TÜİK MEDAS exports
`il_nufus_verisi.csv` (`Adana-1`) and `ilce_nufus_verisi.csv` (`Adana(Aladağ)-1757`).
Every other district-level table can be joined on `district_code` (or `city_code` for provinces) to get consistent names and codes.
The table reflects the 2025 administrative structure; it covers every district that exists from 2013 onward.
Input and output live in the same folder (`data/raw/`); `district_name` is **not** unique (e.g. 51 × "Merkez"), only `district_code` is.

In [ ]:
# Central reference table: city_code | city_name | district_code | district_name  (join key for every district-level table)
import os
import pandas as pd

DATA_DIR = "../data/raw"  # inputs and output share this folder
CITY_PATH = os.path.join(DATA_DIR, "il_nufus_verisi.csv")
DISTRICT_PATH = os.path.join(DATA_DIR, "ilce_nufus_verisi.csv")
CODE_TABLE_PATH = os.path.join(DATA_DIR, "turkey_city_district_code_table.csv")

# 'Adana-1'  /  'Adana(Aladağ)-1757'  ->  name(s) + code
CITY_LABEL_PATTERN = r'^(?P<city_name>.*)-(?P<city_code>\d+)$'
DISTRICT_LABEL_PATTERN = r'^(?P<city_name>.*?)\((?P<district_name>.*)\)-(?P<district_code>\d+)$'


def _parse_labels(path: str, pattern: str) -> pd.DataFrame:
    """MEDAS pivot export -> one row per 'Name-Code' label; header/separator rows are NaN and dropped."""
    labels = pd.read_csv(
        path, sep='|', header=None, usecols=[1], names=['label'], encoding='utf-8-sig',
    )['label'].dropna().str.strip()
    parsed = labels.str.extract(pattern)
    if parsed.isna().any(axis=None):
        raise ValueError(f"Unparseable labels in {path}: {labels[parsed.isna().any(axis=1)].tolist()}")
    return parsed


def build_district_code_table(city_path: str, district_path: str) -> pd.DataFrame:
    cities = _parse_labels(city_path, CITY_LABEL_PATTERN)
    districts = _parse_labels(district_path, DISTRICT_LABEL_PATTERN)

    # The district labels carry the city *name*; the city code comes from the city file.
    table = districts.merge(cities, on='city_name', how='left', validate='many_to_one')
    if table['city_code'].isna().any():
        raise ValueError(f"Cities missing from {city_path}: {sorted(table.loc[table['city_code'].isna(), 'city_name'].unique())}")
    if table['district_code'].duplicated().any():
        raise ValueError(f"Duplicate district_code: {sorted(table.loc[table['district_code'].duplicated(), 'district_code'].unique())}")

    return (
        table.astype({'city_code': int, 'district_code': int})[['city_code', 'city_name', 'district_code', 'district_name']]
        .sort_values(['city_code', 'district_code'], ignore_index=True)
    )


df_codes = build_district_code_table(CITY_PATH, DISTRICT_PATH)
df_codes.to_csv(CODE_TABLE_PATH, index=False)
print(f"{len(df_codes)} districts across {df_codes['city_code'].nunique()} cities -> {CODE_TABLE_PATH}")
df_codes.head(5)


# District-Level Preprocessing Pipeline (GeoPandas & TÜİK ID Integration)

This notebook establishes the spatial data preprocessing pipeline for the **AI-Driven Urban Green Planner**.
It uses **GeoPandas** to:
1. Ingest raw UN OCHA administrative boundaries from HDX (districts and cities), keeping only the columns we need.
2. Join them to the central code table built above: `city_code` (secondary key) from the OCHA P-code, `district_code` (primary key) from a name match inside each city.
3. Validate the joins (strict 1:1) and the spatial centroids (100% Point-in-Polygon check).
4. Export the single sources of truth: `turkey_districts_master.geojson` and `turkey_cities_master.geojson`.
5. Render spatial verification boundary maps.

## 1. Imports and Paths Configuration
We use `geopandas`, `pandas`, `shapely`, and `matplotlib` within the `general_use` conda environment.

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

RAW_DISTRICTS_PATH = "../data/raw/turkey_districts_raw.geojson"
RAW_CITIES_PATH = "../data/raw/turkey_provinces_raw.geojson"
CODE_TABLE_PATH = "../data/raw/turkey_city_district_code_table.csv"
PROCESSED_GEOJSON_PATH = "../data/processed/turkey_districts_master.geojson"
PROCESSED_CITIES_GEOJSON_PATH = "../data/processed/turkey_cities_master.geojson"

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../data/results", exist_ok=True)
print("GeoPandas environment configured.")


## 2. Ingest Boundaries & Join the Central Code Table
Load the raw district and city boundaries (`columns=` drops the constant/empty OCHA attributes at read time) and join both to the central code table:
- **Districts**, on **both** keys in a single merge:
  - **City:** `city_code` is the plate code inside the OCHA `adm1_pcode` (`TUR034` → `34`).
  - **District:** a normalized `district_name` (accents, case, punctuation) matched **within the city**, so the 51 × "Merkez" never collide. OCHA names the central district of non-metropolitan cities after the city (`Adiyaman`), so the code table's `Merkez` rows are keyed by their city name.
- **Cities:** `city_code` from `adm1_pcode`, names from the code table; the OCHA city name is only a sanity check of the plate-code extraction.

Both merges are strictly 1:1 in both directions and raise if any polygon or code is left over (this also catches a wrong city code).

In [ ]:
# `columns=` drops everything we do not need (constant/empty OCHA attributes: adm0_*, lang*, version, valid_*, ...) at read time
DISTRICT_COLUMNS = ["adm2_name1", "adm1_pcode", "area_sqkm", "center_lat", "center_lon"]
CITY_COLUMNS = ["adm1_pcode", "adm1_name", "area_sqkm", "center_lat", "center_lon"]

gdf_raw = gpd.read_file(RAW_DISTRICTS_PATH, columns=DISTRICT_COLUMNS)
gdf_cities_raw = gpd.read_file(RAW_CITIES_PATH, columns=CITY_COLUMNS)
df_codes = pd.read_csv(CODE_TABLE_PATH)

print(f"Raw district boundaries loaded: {len(gdf_raw)} features.")
print(f"Raw city boundaries loaded: {len(gdf_cities_raw)} features.")
print(f"Central code table loaded: {len(df_codes)} districts in {df_codes['city_code'].nunique()} cities.")


In [ ]:
def name_key(names: pd.Series) -> pd.Series:
    """Accent/case/punctuation-insensitive match key: 'Çukurova' and 'Cukurova' -> 'cukurova', 'Iğdır' -> 'igdir'."""
    return names.str.normalize('NFKD').str.lower().str.replace('ı', 'i').str.replace(r'[^a-z0-9]', '', regex=True)


# OCHA side: the city plate code sits inside the P-code ('TUR034' -> 34)
ocha = gdf_raw.assign(city_code=gdf_raw['adm1_pcode'].str[-3:].astype(int), key=name_key(gdf_raw['adm2_name1']))
# Code table side: TÜİK calls the central district of a non-metropolitan city 'Merkez', OCHA names it after the city
codes = df_codes.assign(key=name_key(df_codes['district_name'].where(df_codes['district_name'] != 'Merkez', df_codes['city_name'])))

# Strict 1:1 in both directions (this also catches a wrong city_code): raises if any polygon or code is left over
merged = ocha.merge(codes, on=['city_code', 'key'], how='outer', validate='one_to_one', indicator=True)
if (merged['_merge'] != 'both').any():
    raise ValueError(merged.loc[merged['_merge'] != 'both', ['_merge', 'city_code', 'adm2_name1', 'district_code', 'district_name']].to_string(index=False))

gdf = (
    merged[['district_code', 'city_code', 'district_name', 'city_name', 'area_sqkm', 'center_lat', 'center_lon', 'geometry']]
    .round({'area_sqkm': 2, 'center_lat': 6, 'center_lon': 6})
    .sort_values(['city_code', 'district_code'], ignore_index=True)
)

# Point-in-polygon verification (vectorized; 'intersects' = inside or on the boundary)
centroids = gpd.GeoSeries(gpd.points_from_xy(gdf['center_lon'], gdf['center_lat']), index=gdf.index, crs=gdf.crs)
print(f"Spatial Validation: {gdf.geometry.intersects(centroids).sum()} / {len(gdf)} centroids inside district boundaries.")
print(f"Code Validation: {len(gdf)} / {len(gdf_raw)} districts joined to the central code table (strict 1:1).")


In [ ]:
# Cities: same plate-code extraction, names from the code table. The OCHA name is only a sanity check of the extraction.
gdf_cities = gdf_cities_raw.assign(city_code=gdf_cities_raw['adm1_pcode'].str[-3:].astype(int)).merge(
    df_codes[['city_code', 'city_name']].drop_duplicates(),
    on='city_code', how='outer', validate='one_to_one', indicator=True,
)
bad = (gdf_cities['_merge'] != 'both') | (name_key(gdf_cities['adm1_name']) != name_key(gdf_cities['city_name']))
if bad.any():
    raise ValueError(gdf_cities.loc[bad, ['_merge', 'city_code', 'adm1_pcode', 'adm1_name', 'city_name']].to_string(index=False))

gdf_cities = (
    gdf_cities[['city_code', 'city_name', 'area_sqkm', 'center_lat', 'center_lon', 'geometry']]
    .round({'area_sqkm': 2, 'center_lat': 6, 'center_lon': 6})
    .sort_values('city_code', ignore_index=True)
)

centers = gpd.GeoSeries(gpd.points_from_xy(gdf_cities['center_lon'], gdf_cities['center_lat']), index=gdf_cities.index, crs=gdf_cities.crs)
area_gap = (gdf.groupby('city_code')['area_sqkm'].sum() - gdf_cities.set_index('city_code')['area_sqkm']).abs().max()
print(f"City Validation: {len(gdf_cities)} cities joined, {gdf_cities.geometry.intersects(centers).sum()} / {len(gdf_cities)} centers inside their polygon, "
      f"max |sum of district areas - city area| = {area_gap:.2f} sq km.")


## 3. Export Master GeoJSON Datasets
Save `turkey_districts_master.geojson` (973 district polygons, centroids, the central keys `district_code` primary / `city_code` secondary, official TÜİK names) and `turkey_cities_master.geojson` (81 city polygons, centers, `city_code`, official TÜİK names).

In [ ]:
gdf.to_file(PROCESSED_GEOJSON_PATH, driver="GeoJSON")
gdf_cities.to_file(PROCESSED_CITIES_GEOJSON_PATH, driver="GeoJSON")
print(f"Saved Master GeoJSON to {PROCESSED_GEOJSON_PATH} ({os.path.getsize(PROCESSED_GEOJSON_PATH) / (1024*1024):.2f} MB)")
print(f"Saved Cities GeoJSON to {PROCESSED_CITIES_GEOJSON_PATH} ({os.path.getsize(PROCESSED_CITIES_GEOJSON_PATH) / (1024*1024):.2f} MB)")
print("Sample Master Records:")
print(gdf.drop(columns="geometry").head(5).to_string(index=False))
print("Sample City Records:")
print(gdf_cities.drop(columns="geometry").head(3).to_string(index=False))


## 4. GeoPandas Direct Map Visualization
Render all 973 district boundaries and center coordinates using GeoPandas built-in plotting.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
gdf.plot(ax=ax, color="#e5f5e0", edgecolor="#31a354", linewidth=0.4)
ax.scatter(gdf["center_lon"], gdf["center_lat"], color="#006d2c", s=1.5, alpha=0.7, label="District Centroids")
ax.set_title("Türkiye 973 District Administrative Boundaries & Centroids (with TÜİK IDs)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.legend(loc="lower right")
plt.tight_layout()
overview_path = "../data/results/turkey_districts_overview.png"
plt.savefig(overview_path, dpi=200, bbox_inches="tight")
print(f"District boundary plot saved to {overview_path}")
plt.show()